# Solution to p11

The cross-entropy is first computed per pixel and then averaged only over the valid mask. Hard foreground Dice likewise masks both the prediction and truth before counting intersection and foreground sizes.

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.tensor([[[[2.,-1.],[0.,1.]], [[0.,2.],[1.,-1.]]]], dtype=torch.float32)
target = torch.tensor([[[0,1],[1,0]]], dtype=torch.int64)
valid = torch.tensor([[[True,True],[True,False]]], dtype=torch.bool)
eps = 1e-7
ATOL = 1e-6
RTOL = 1e-6

def masked_pixel_ce_and_dice(logits, target, valid):
    if logits.ndim != 4 or logits.shape[1] != 2 or target.shape != logits.shape[:1]+logits.shape[2:] or valid.shape != target.shape:
        raise ValueError("shape mismatch")
    if logits.dtype != torch.float32 or target.dtype != torch.int64 or valid.dtype != torch.bool:
        raise ValueError("dtype mismatch")
    if logits.device.type != "cpu" or target.device.type != "cpu" or valid.device.type != "cpu":
        raise ValueError("inputs must be on CPU")
    if not bool(valid.any()):
        raise ValueError("at least one pixel must be valid")
    per_pixel = F.cross_entropy(logits, target, reduction="none")
    mean_ce = per_pixel[valid].mean()
    prediction = logits.argmax(dim=1)
    pred_fg = (prediction == 1) & valid
    true_fg = (target == 1) & valid
    intersection = (pred_fg & true_fg).sum(dtype=torch.float32)
    dice = (2*intersection + eps) / (pred_fg.sum(dtype=torch.float32) + true_fg.sum(dtype=torch.float32) + eps)
    return mean_ce, dice

mean_ce, dice = masked_pixel_ce_and_dice(logits, target, valid)


### Answer check

In [ ]:
assert mean_ce.ndim == 0 and dice.ndim == 0
torch.testing.assert_close(mean_ce, torch.tensor(0.16292568), atol=ATOL, rtol=RTOL)
torch.testing.assert_close(dice, torch.tensor(1.0), atol=ATOL, rtol=RTOL)
try:
    masked_pixel_ce_and_dice(logits, target, torch.zeros_like(valid))
except ValueError:
    pass
else:
    raise AssertionError("empty valid mask was not rejected")
